# Graded Response Model — Single Scale

This notebook fits a standard **GRModel** (single latent dimension) to all 22
RWA items, treating the entire questionnaire as measuring one underlying
construct. Compare with `factorized_grm_missing.ipynb` which splits items
into two subscales via `FactorizedGRModel`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

## 1. Load the RWA Dataset

In [ ]:
from bayesianquilts.data.rwa import get_data, item_keys, item_text

df, num_people = get_data(polars_out=True)
print(f"Dataset: {num_people} people, {len(item_keys)} items, 9 response categories (0-8)")
df.head()

In [ ]:
SUBSAMPLE_N = num_people
sub_df = df
print(f"Using full dataset: N = {SUBSAMPLE_N}")

## 2. Prepare Data and Fit the Single-Scale GRM

We use `GRModel` with `dim=1` so that all 22 items load on a single
latent trait. Missing/invalid responses (if any) have their
log-likelihood zeroed out.

In [ ]:
def make_data_dict(dataframe):
    """Convert polars DataFrame to dict of numpy float64 arrays."""
    data = {}
    for col in dataframe.columns:
        arr = dataframe[col].to_numpy().astype(np.float64)
        data[col] = arr
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

batch = make_data_dict(sub_df)

# Check for missing/invalid values
n_bad = sum(
    np.sum(np.isnan(batch[k]) | (batch[k] < 0) | (batch[k] >= 9))
    for k in item_keys
)
print(f"Bad/missing values: {n_bad}")

BATCH_SIZE = 256
steps_per_epoch = int(np.ceil(SUBSAMPLE_N / BATCH_SIZE))
print(f"N: {SUBSAMPLE_N}, Batch size: {BATCH_SIZE}, Steps per epoch: {steps_per_epoch}")

def data_factory():
    indices = np.arange(SUBSAMPLE_N)
    np.random.shuffle(indices)
    for start in range(0, SUBSAMPLE_N, BATCH_SIZE):
        end = min(start + BATCH_SIZE, SUBSAMPLE_N)
        idx_batch = indices[start:end]
        yield {k: v[idx_batch] for k, v in batch.items()}

In [ ]:
from bayesianquilts.irt.grm import GRModel

model = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    kappa_scale=0.1,
    response_cardinality=9,
    dtype=jnp.float64,
)

NUM_EPOCHS = 200

losses, params = model.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
)

print(f"Final loss: {losses[-1]:.2f}")

## 3. Training Diagnostics

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss (neg ELBO)')
plt.title('Training Loss — Single-Scale GRM')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Posterior Parameter Estimates

In [ ]:
def calibrate_manually(model, n_samples=32, seed=42):
    surrogate = model.surrogate_distribution_generator(model.params)
    key = jax.random.PRNGKey(seed)
    samples = surrogate.sample(n_samples, seed=key)
    expectations = {}
    for k, v in samples.items():
        expectations[k] = jnp.mean(v, axis=0)
    model.calibrated_expectations = expectations
    model.surrogate_sample = samples

calibrate_manually(model, n_samples=32, seed=101)

In [ ]:
disc = np.array(model.calibrated_expectations['discriminations']).flatten()

fig, ax = plt.subplots(figsize=(8, 6))
y_pos = np.arange(len(item_keys))
ax.barh(y_pos, disc, alpha=0.7, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(item_keys)
ax.set_xlabel('Discrimination')
ax.set_title('Item Discriminations — Single Scale')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
abilities = np.array(model.calibrated_expectations['abilities']).flatten()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(abilities, bins=30, alpha=0.7, edgecolor='black')
ax.set_xlabel('Ability (RWA latent trait)')
ax.set_ylabel('Count')
ax.set_title('Ability Distribution — Single Scale')
plt.tight_layout()
plt.show()

In [ ]:
diff0 = np.array(model.calibrated_expectations['difficulties0'])  # (1, D, I, 1)
ddiff = np.array(model.calibrated_expectations['ddifficulties'])   # (1, D, I, K-2)

# Reconstruct cumulative thresholds
d0 = np.concatenate([diff0, ddiff], axis=-1)  # (1, D, I, K-1)
thresholds = np.cumsum(d0, axis=-1)[0, 0, :, :]  # (I, K-1)

fig, ax = plt.subplots(figsize=(10, 6))
for i, key in enumerate(item_keys):
    ax.plot(range(1, 9), thresholds[i], 'o-', alpha=0.5, markersize=3, label=key)
ax.set_xlabel('Threshold index')
ax.set_ylabel('Threshold value')
ax.set_title('Item Difficulty Thresholds')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated fitting a single-scale Graded Response Model to
all 22 RWA items using `GRModel` with `dim=1`. All items load on one latent
trait, in contrast to the factorized model which partitions items across two
subscales. The discrimination parameters indicate how well each item
differentiates respondents along this single RWA dimension.